# Day 054 — Exercise 1: Schema & Creating Conversations

**What you'll build:** `create_schema(engine)` and `create_conversation(session, title)` over the provided `Conversation` and `Message` models (a one-to-many relationship).

**Why it matters:** So far the chat app forgot everything on restart. Today it gets a database. The models define two related tables — a `Conversation` has many `Message`s — and `create_schema` builds them. `create_conversation` inserts a row and flushes so the database assigns its primary key. This is the foundation of saved state.

## Provided: Setup + Models (Conversation 1--* Message)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import tempfile
from datetime import datetime
from sqlalchemy import create_engine, ForeignKey, select, inspect as sa_inspect, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Conversation(Base):
    """One chat conversation. Has many Messages (one-to-many)."""
    __tablename__ = 'conversations'

    id:         Mapped[int]      = mapped_column(primary_key=True)
    title:      Mapped[str]      = mapped_column(default='New chat')
    created_at: Mapped[datetime] = mapped_column(default=datetime.utcnow)

    # relationship() is the ORM link (not a DB column). cascade deletes a
    # conversation's messages when the conversation is deleted.
    messages: Mapped[list['Message']] = relationship(
        back_populates='conversation', cascade='all, delete-orphan')


class Message(Base):
    """One message in a conversation. Belongs to one Conversation (many-to-one)."""
    __tablename__ = 'messages'

    id:              Mapped[int]      = mapped_column(primary_key=True)
    conversation_id: Mapped[int]      = mapped_column(ForeignKey('conversations.id'))
    role:            Mapped[str]      = mapped_column()
    content:         Mapped[str]      = mapped_column()
    created_at:      Mapped[datetime] = mapped_column(default=datetime.utcnow)

    conversation: Mapped['Conversation'] = relationship(back_populates='messages')


def memory_engine():
    """In-memory SQLite engine for tests. StaticPool makes every Session share the
    one in-memory database (see Day 44)."""
    return create_engine('sqlite:///:memory:',
                          connect_args={'check_same_thread': False},
                          poolclass=StaticPool)

## Your Implementation

In [ ]:
def create_schema(engine) -> None:
    """Create every table registered on Base."""
    # TODO: Base.metadata.create_all(engine)
    pass


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a Conversation, flush to assign its id, and return it.
    The caller commits."""
    # TODO: conv = Conversation(title=title)
    # TODO: session.add(conv)
    # TODO: session.flush()   # assigns conv.id without committing
    # TODO: return conv
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    engine = memory_engine()

    # Check 1: create_schema builds both tables
    try:
        create_schema(engine)
        tables = set(sa_inspect(engine).get_table_names())
        assert {'conversations', 'messages'} <= tables, f'missing tables: {tables}'
        passed += 1; print('✅ Check 1: create_schema builds conversations + messages')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: create_conversation assigns an id after flush
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'My first chat')
            assert conv.id is not None, 'id should be assigned after flush'
            s.commit()
        passed += 1; print('✅ Check 2: create_conversation assigns an id')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: title is stored
    try:
        with Session(engine) as s:
            conv = create_conversation(s, 'Weather bot')
            s.commit()
            assert conv.title == 'Weather bot', f'title not stored: {conv.title}'
        passed += 1; print('✅ Check 3: title is persisted')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: default title + created_at timestamp
    try:
        with Session(engine) as s:
            conv = create_conversation(s)
            s.commit()
            assert conv.title == 'New chat', f'default title wrong: {conv.title}'
            assert isinstance(conv.created_at, datetime), 'created_at should be a datetime'
        passed += 1; print('✅ Check 4: default title + created_at set')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: distinct conversations get distinct ids
    try:
        with Session(engine) as s:
            a = create_conversation(s, 'A')
            b = create_conversation(s, 'B')
            s.commit()
            assert a.id != b.id, 'ids must be unique'
        passed += 1; print('✅ Check 5: each conversation gets a unique id')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def create_schema(engine) -> None:
    """Create every table registered on Base (CREATE TABLE IF NOT EXISTS)."""
    Base.metadata.create_all(engine)


def create_conversation(session, title: str = 'New chat') -> Conversation:
    """Insert a new conversation and flush so its auto id is assigned.
    The caller controls commit (unit-of-work pattern)."""
    conv = Conversation(title=title)
    session.add(conv)
    session.flush()
    return conv
```

**Why this works:** `Base.metadata.create_all(engine)` emits `CREATE TABLE IF NOT EXISTS` for every model registered on `Base` — both tables at once. `create_conversation` adds the object and calls `session.flush()`, which sends the INSERT and lets the database assign the auto-increment `id` — without committing. Returning the object with its id lets the caller attach messages to it. The caller owns the commit, so several operations can share one transaction.
</details>